# Parte 1 — Análise Exploratória dos Chamados (Q1)

## Contexto

A **Central 1746** recebe chamados de cidadãos que precisam ser encaminhados para o serviço correto. O **modelo A** (em produção) e o **modelo B** (candidato) classificam automaticamente o texto do chamado em categorias de serviço.

Este notebook explora o corpus de **5.000 chamados rotulados** (`dados/chamados_com_predicoes.csv`) para entender características dos dados que **impactam a avaliação dos classificadores**.

> **Nota:** os dados são sintéticos — simulam o comportamento estatístico de um sistema real, sem uso de dados de cidadãos.

## Objetivo

Identificar padrões no corpus (categorias, textos, canal, etc.) relevantes para o **problema de classificação**, preparando a auditoria do modelo A (Parte 2) e a comparação A vs B (Parte 3).

## Dados utilizados

| Coluna | Descrição |
|--------|-----------|
| `texto` | Texto do chamado (input dos modelos) |
| `categoria_real` | Rótulo humano (ground truth) |
| `canal`, `bairro`, `data_abertura` | Metadados do chamado |
| `pred_modelo_a/b`, `conf_modelo_a/b` | Predições dos modelos *(analisadas nas Partes 2 e 3)* |

## Roteiro da análise

1. **Panorama geral** — dimensões, qualidade e período dos dados  
2. **Distribuição de categorias** — desbalanceamento entre classes  
3. **Características dos textos** — comprimento e variabilidade do input  
4. **Padrões por canal** — diferenças na forma de entrada dos chamados  
5. **Síntese** — 3–5 achados principais e relevância para classificação

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "dados" / "chamados_com_predicoes.csv"
FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Carregamento e validação inicial

Carregamos o CSV, validamos dimensões e ausência de nulos, e criamos
colunas derivadas para as análises seguintes.

In [ ]:
# Carregando o arquivo CSV
df = pd.read_csv(DATA_PATH)

# Colunas derivadas para as análises seguintes
df["texto_len"] = df["texto"].str.len()
df["data_abertura"] = pd.to_datetime(df["data_abertura"])

# Sanity checks (estrutura e domínio)
assert df.shape == (5000, 11)
assert df.isna().sum().sum() == 0
assert (df["id_chamado"]).is_unique
assert (df["texto_len"] > 0).all()
assert (df["conf_modelo_a"]).between(0, 1).all()
assert (df["conf_modelo_b"]).between(0, 1).all()

# Exibindo estrutura e as primeiras linhas do DataFrame
df.info()
display(df.head())

## 1. Panorama geral

Antes de analisar textos e modelos, verificamos a **estrutura e a qualidade** do corpus:

- **Volume e integridade:** número de registros, colunas, valores ausentes e IDs duplicados.
- **Cobertura temporal:** intervalo de `data_abertura`.
- **Cardinalidade:** quantas categorias, canais e bairros distintos existem.
- **Primeira visão de distribuição:** contagem por **canal** e por **`categoria_real`**, com proporções e razão entre a classe mais e a menos frequente.

Essas checagens contextualizam o problema de **classificação**: dados limpos permitem avaliar os modelos com confiança; desbalanceamento e viés de canal antecipam onde a acurácia global pode esconder erros.

In [ ]:
resumo = pd.Series({
    "registros": len(df),
    "colunas": df.shape[1],
    "ids_duplicados": df.duplicated(subset=["id_chamado"]).sum(),
    "valores_nulos": df.isna().sum().sum(),
    "periodo_inicio": df["data_abertura"].min().date(),
    "periodo_fim": df["data_abertura"].max().date(),
    "n_categorias": df["categoria_real"].nunique(),
    "n_canais": df["canal"].nunique(),
    "n_bairros": df["bairro"].nunique(),
})

display(resumo)


print("\nContagem por canal:")
display(df["canal"].value_counts())

cat = df["categoria_real"].value_counts().to_frame("n")
cat["pct"] = (cat["n"] / len(df) * 100).round(1)
display(cat)
print(f"Razão maior/menor classe: {cat['n'].max() / cat['n'].min():.1f}:1")

### Interpretação

**Qualidade e escopo**
- **5.000 chamados** no CSV original (**10 colunas**), mais a coluna derivada `texto_len` (**11 no total**). **0 nulos** e **0 IDs duplicados** — corpus consistente, sem tratamento de missingness nesta etapa.
- Período **2026-01-01** a **2026-06-30** (~6 meses), com **8 categorias**, **3 canais** e **15 bairros**.

**Canal de entrada**
| Canal | n | % aprox. |
|-------|---|----------|
| `app_1746` | 2.408 | 48,2% |
| `telefone_1746` | 1.724 | 34,5% |
| `portal_web` | 868 | 17,4% |

O **app** concentra quase metade dos chamados. Isso pode influenciar comprimento e qualidade do texto — aprofundamos na **§4**.

**Categorias (`categoria_real`)**
- A tabela acima mostra **volumes desiguais** entre as 8 classes (razão **maior/menor: 5,0:1**).
- O desbalanceamento e sua implicação para os classificadores são **visualizados e discutidos na §2**.

**Relevância para classificação (panorama)**
1. Base limpa → métricas dos modelos A e B refletem desempenho real, não ruído de dados.
2. Concentração por canal → possível **efeito canal** na classificação; validar na **§4**.

**Próximo passo:** §2 — gráfico da distribuição de categorias.

## 2. Distribuição de categorias

No §1 quantificamos `categoria_real` em tabela (contagens, % e razão 5,0:1). Aqui **visualizamos** essa distribuição.

**O que o gráfico acrescenta**
- Leitura rápida da **concentração no topo** vs. **cauda** das classes.
- Destaque visual da classe minoritária (**Sinalização**, 4,5%).
- Figura exportada para o relatório (`results/figures/`).

**Próximo passo após esta seção:** interpretar implicações para a avaliação dos classificadores (Parte 2).

In [ ]:
LABEL_CAT = {
    "iluminacao_publica": "Iluminação pública",
    "buraco_via": "Buraco em via",
    "coleta_lixo": "Coleta de lixo",
    "esgoto_vazamento": "Esgoto / vazamento",
    "poda_arvore": "Poda de árvore",
    "estacionamento_irregular": "Estacionamento irregular",
    "barulho_perturbacao": "Barulho / perturbação",
    "sinalizacao": "Sinalização",
}

# Reutiliza `cat` criada no §1
cat_plot = cat.sort_values("n", ascending=False).reset_index()
cat_plot["label"] = cat_plot["categoria_real"].map(LABEL_CAT)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(
    data=cat_plot,
    y="label",
    x="n",
    ax=ax,
    color="steelblue",
)

ax.set_xlabel("Número de chamados")
ax.set_ylabel("")
ax.set_title("")  # título vai no nível da figura (evita overlap)

fig.suptitle(
    "Distribuição dos chamados por tipo de serviço",
    fontsize=13,
    fontweight="bold",
    x=0.06,
    ha="left",
    y=0.98,
)
fig.text(
    0.06, 0.91,
    "Central 1746 · 5.000 chamados · jan–jun/2026 · dados sintéticos",
    fontsize=9,
    color="gray",
    ha="left",
    transform=fig.transFigure,
)

# Espaço para os % à direita das barras
xmax = cat_plot["n"].max()
ax.set_xlim(0, xmax * 1.18)

# Rótulos de % alinhados às barras
for bar, (_, row) in zip(ax.patches, cat_plot.iterrows()):
    ax.text(
        bar.get_width() + 12,
        bar.get_y() + bar.get_height() / 2,
        f"{row['pct']:.1f}%",
        va="center",
        ha="left",
        fontsize=10,
    )

sns.despine(ax=ax, left=False)

fig.tight_layout(rect=[0, 0, 1, 0.88])  # reserva topo para título + subtítulo
fig.savefig(FIG_DIR / "01_distribuicao_categorias.png", dpi=200, bbox_inches="tight")
plt.show()




### Interpretação

A figura confirma o desbalanceamento já reportado no §1, agora de forma **visual**:

- **Concentração no topo:** três tipos de serviço (Iluminação pública, Buraco em via, Coleta de lixo) respondem por **~57%** dos chamados — barras claramente mais longas que as demais.
- **Cauda intermediária:** cinco categorias ficam entre **~8% e ~12%**, com volumes parecidos entre si.
- **Minoritária destacada:** **Sinalização** (4,5%) isola-se na base do gráfico — razão **5,0:1** vs. a majoritária (1.143 vs 227 chamados).

**Relevância para classificação**
1. **Viés de frequência:** acertar bem as classes do topo pode inflar a acurácia global, mesmo com erros sistemáticos em **Sinalização** e outras da cauda.
2. **Métricas na Parte 2:** reportar **recall e F1 por categoria** (e F1 macro), além da acurácia agregada.
3. **Hipótese:** classes com menos exemplos tendem a **recall menor** — validar na matriz de confusão do modelo A.

**Achado provisório (síntese final):** desbalanceamento moderado entre 8 categorias → avaliar modelos **por classe**, não só no agregado.

Figura: `results/figures/01_distribuicao_categorias.png`

**Próximo passo:** §3 — características dos textos (`texto_len`).

## 3. Características dos textos

Os classificadores A e B recebem apenas o campo **`texto`** como entrada. A variável **`texto_len`** (criada no carregamento) mede quantos caracteres cada chamado traz — proxy da **quantidade de contexto** disponível para a predição.

**O que esta seção responde**
- Qual é a **variabilidade global** do comprimento dos textos?
- Alguma **categoria** tende a textos mais curtos ou mais longos?
- Como se parecem **exemplos extremos** (curto vs longo)?

**Por que importa para classificação**
Textos curtos costumam ter **menos pistas léxicas** (endereço, detalhe, contexto), o que pode reduzir acurácia — hipótese a validar na Parte 2 (subgrupos por faixa de comprimento).

**Próximo passo após esta seção:** interpretar padrões e exportar figuras para `results/figures/`.

In [ ]:
# --- Resumo numérico ---
print("Comprimento do texto (caracteres):")
display(df["texto_len"].describe().round(1))

# Faixas úteis para a Parte 2
faixas = pd.cut(
    df["texto_len"],
    bins=[0, 60, 149, 200],
    labels=["curto (≤60)", "médio (61–149)", "longo (≥150)"],
)
print("\nDistribuição por faixa:")
display(faixas.value_counts().to_frame("n").assign(pct=lambda x: (x["n"] / len(df) * 100).round(1)))

# --- Figura 1: histograma global ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["texto_len"], bins=30, kde=True, ax=ax, color="steelblue")

ax.set_xlabel("Comprimento do texto (caracteres)")
ax.set_ylabel("Número de chamados")
ax.set_title("")

fig.suptitle(
    "Distribuição do comprimento dos textos dos chamados",
    fontsize=13, fontweight="bold", x=0.06, ha="left", y=0.98,
)
fig.text(
    0.06, 0.91,
    "Central 1746 · 5.000 chamados · jan–jun/2026 · dados sintéticos",
    fontsize=9, color="gray", ha="left", transform=fig.transFigure,
)

sns.despine(ax=ax)
fig.tight_layout(rect=[0, 0, 1, 0.88])
fig.savefig(FIG_DIR / "02_distribuicao_texto_len.png", dpi=200, bbox_inches="tight")
plt.show()

# --- Figura 2: mediana por categoria ---
len_cat = (
    df.groupby("categoria_real")["texto_len"]
    .agg(mediana="median", n="count")
    .reset_index()
    .sort_values("mediana", ascending=False)
)
len_cat["label"] = len_cat["categoria_real"].map(LABEL_CAT)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=len_cat, y="label", x="mediana", ax=ax, color="coral")

ax.set_xlabel("Mediana de caracteres")
ax.set_ylabel("")
ax.set_title("")

fig.suptitle(
    "Comprimento mediano do texto por tipo de serviço",
    fontsize=13, fontweight="bold", x=0.06, ha="left", y=0.98,
)
fig.text(
    0.06, 0.91,
    "Central 1746 · mediana de caracteres por categoria",
    fontsize=9, color="gray", ha="left", transform=fig.transFigure,
)

xmax = len_cat["mediana"].max()
ax.set_xlim(0, xmax * 1.12)
for bar, (_, row) in zip(ax.patches, len_cat.iterrows()):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{row['mediana']:.0f}",
        va="center", ha="left", fontsize=10,
    )

sns.despine(ax=ax, left=False)
fig.tight_layout(rect=[0, 0, 1, 0.88])
fig.savefig(FIG_DIR / "03_texto_len_por_categoria.png", dpi=200, bbox_inches="tight")
plt.show()

# --- Exemplos curtos vs longos ---
cols = ["texto_len", "categoria_real", "texto"]
print("\nExemplos mais curtos:")
display(df.nsmallest(3, "texto_len")[cols])
print("\nExemplos mais longos:")
display(df.nlargest(3, "texto_len")[cols])

### Interpretação

**Distribuição global** (fig. `02_distribuicao_texto_len.png`): o comprimento é **bimodal** — pico de textos **curtos** (~35–50 caracteres, **28,4%** ≤60) e pico de textos **longos** (~155–165, **62,1%** ≥150), com poucos no meio (**9,4%**). A **média (128)** cai nesse vale; a **mediana (156)** descreve melhor o bloco majoritário.

**Por categoria** (fig. `03_texto_len_por_categoria.png`): medianas entre **147 e 161** — barras quase iguais. A variabilidade de comprimento é **do corpus**, não de um tipo de serviço específico.

**Exemplos:** textos mínimos (24 caracteres, ex. *“Asfalto cedeu há 3 dias.”*) vs. textos detalhados (199 caracteres, com endereço e contexto) ilustram o contraste de **pistas léxicas** disponíveis ao modelo.

**Relevância para classificação**
1. ~**1 em 4** chamados traz pouco texto → hipótese de **maior taxa de erro** nessa faixa (validar na Parte 2).
2. Como todas as categorias têm medianas parecidas, o subgrupo de risco é **comprimento**, não classe — cruzar com **canal** na **§4**.

**Achado provisório:** input **bimodal** (curto vs longo) espalhado por várias classes → testar desempenho do modelo A por faixa de `texto_len`.

**Próximo passo:** §4 — padrões por `canal`.